# Hybrid Retrieval + RAG Fusion (LangGraph)

This notebook implements the **retrieval + reflection** section:
- Hybrid retrieval (BM25 + semantic) with metadata filtering
- Optional re-ranking (non‑LLM)
- Reflection (Good LLM) for relevance check
- Retry with RAG fusion if the first pass is not relevant

Other parts of the full agent (routing, generation, persona, chat history) are intentionally **left out**.

## Install dependencies

In [1]:
%pip -q install -U langgraph qdrant-client sentence-transformers rank-bm25 python-dotenv
%pip -q install -U langchain-core langchain-openai langchain-ollama langchain-huggingface huggingface_hub

## Config (.env + LLM factory)

Create a `.env` file alongside this notebook with two model tiers. You can use **OpenAI**, **Ollama**, or **Hugging Face**.

Example (Hugging Face like the benchmark):
```
LIGHT_LLM_PROVIDER=huggingface
LIGHT_LLM_MODEL=meta-llama/Llama-3.1-8B-Instruct

GOOD_LLM_PROVIDER=huggingface
GOOD_LLM_MODEL=Qwen/Qwen2.5-72B-Instruct

HUGGINGFACEHUB_API_TOKEN=...
HF_PROVIDER=auto
HF_MAX_NEW_TOKENS=1024
HF_TEMPERATURE=0.2
HF_TOP_P=0.95
```

Example (OpenAI + Ollama):
```
LIGHT_LLM_PROVIDER=ollama
LIGHT_LLM_MODEL=llama3.1

GOOD_LLM_PROVIDER=openai
GOOD_LLM_MODEL=gpt-4o-mini
OPENAI_API_KEY=...

OLLAMA_BASE_URL=http://localhost:11434
```

Only the retrieval and reflection steps below use these. If no keys are provided, the notebook falls back to heuristic checks.

In [2]:
from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()


@dataclass
class LLMConfig:
    provider: str
    model: str


class LLMFactory:
    def __init__(self, light: LLMConfig, good: LLMConfig):
        self._light = light
        self._good = good

    def light(self):
        return _build_llm(self._light)

    def good(self):
        return _build_llm(self._good)


def _build_llm(cfg: LLMConfig):
    provider = (cfg.provider or "").lower().strip()
    if provider == "openai":
        try:
            from langchain_openai import ChatOpenAI
        except Exception as exc:
            raise RuntimeError("Install langchain-openai for OpenAI models") from exc

        return ChatOpenAI(model=cfg.model, temperature=0.0)

    if provider == "ollama":
        try:
            from langchain_ollama import ChatOllama
        except Exception as exc:
            raise RuntimeError("Install langchain-ollama for Ollama models") from exc

        base_url = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
        return ChatOllama(model=cfg.model, base_url=base_url, temperature=0.0)

    if provider in {"huggingface", "hf"}:
        try:
            from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
        except Exception as exc:
            raise RuntimeError("Install langchain-huggingface for HF models") from exc

        # Defaults aligned with .env examples (used if not set)
        os.environ.setdefault("HF_PROVIDER", "auto")
        os.environ.setdefault("HF_MAX_NEW_TOKENS", "1024")
        os.environ.setdefault("HF_TEMPERATURE", "0.2")
        os.environ.setdefault("HF_TOP_P", "0.95")
        os.environ.setdefault("HF_TASK", "text-generation")

        hf_provider = os.environ["HF_PROVIDER"]
        hf_max_new_tokens = int(os.environ["HF_MAX_NEW_TOKENS"])
        hf_temperature = float(os.environ["HF_TEMPERATURE"])
        hf_top_p = float(os.environ["HF_TOP_P"])
        hf_task = os.environ["HF_TASK"]

        endpoint = HuggingFaceEndpoint(
            repo_id=cfg.model,
            task=hf_task,
            provider=hf_provider,
            max_new_tokens=hf_max_new_tokens,
            temperature=hf_temperature,
            top_p=hf_top_p,
            return_full_text=False,
        )
        return ChatHuggingFace(llm=endpoint)

    return None


light_cfg = LLMConfig(
    provider=os.environ.get("LIGHT_LLM_PROVIDER", ""),
    model=os.environ.get("LIGHT_LLM_MODEL", ""),
)

good_cfg = LLMConfig(
    provider=os.environ.get("GOOD_LLM_PROVIDER", ""),
    model=os.environ.get("GOOD_LLM_MODEL", ""),
)

factory = LLMFactory(light=light_cfg, good=good_cfg)
light_llm = factory.light()
good_llm = factory.good()

print("Light LLM:", light_llm.model_id)
print("Good LLM:", good_llm.model_id)
if light_llm is None or good_llm is None:
    print("LLM(s) not configured. Heuristic fallback will be used.")

Light LLM: meta-llama/Llama-3.1-8B-Instruct
Good LLM: Qwen/Qwen2.5-72B-Instruct


## Load data (TO BE CHANGED)

In [3]:
from pathlib import Path
from typing import Any

import pandas as pd

csv_path = Path("/fashion.csv")

if not csv_path.exists():
    raise FileNotFoundError("Expected fashion.csv next to this notebook")

df = pd.read_csv(csv_path).fillna("")
print("Columns:", list(df.columns))

# Use all CSV columns as metadata fields
metadata_fields = list(df.columns)
print("Metadata fields used:", metadata_fields)


def build_document_text(row: pd.Series) -> str:
    title = str(row.get("ProductTitle", "")).strip()
    size = str(row.get("Size", "")).strip()
    parts = []
    if title:
        parts.append(f"ProductTitle: {title}")
    if size:
        parts.append(f"Size: {size}")
    return " | ".join(parts)


documents: list[dict[str, Any]] = []
for idx, row in df.iterrows():
    doc_text = build_document_text(row)
    metadata = {col: str(row[col]).strip() for col in metadata_fields if str(row[col]).strip()}
    documents.append({"id": int(idx), "text": doc_text, "metadata": metadata})

print("Total docs:", len(documents))
print("Sample doc:", documents[0])

Columns: ['ProductId', 'Gender', 'Category', 'SubCategory', 'ProductType', 'Colour', 'Usage', 'ProductTitle', 'Image', 'ImageURL', 'Size']
Metadata fields used: ['ProductId', 'Gender', 'Category', 'SubCategory', 'ProductType', 'Colour', 'Usage', 'ProductTitle', 'Image', 'ImageURL', 'Size']
Total docs: 2906
Sample doc: {'id': 0, 'text': 'ProductTitle: Gini and Jony Girls Knit White Top | Size: Large, X-Small, Small', 'metadata': {'ProductId': '42419', 'Gender': 'Girls', 'Category': 'Apparel', 'SubCategory': 'Topwear', 'ProductType': 'Tops', 'Colour': 'White', 'Usage': 'Casual', 'ProductTitle': 'Gini and Jony Girls Knit White Top', 'Image': '42419.jpg', 'ImageURL': 'https://assets.myntassets.com/v1/images/style/properties/f3964f76c78edd85f4512d98b26d52e9_images.jpg', 'Size': 'Large, X-Small, Small'}}


## Indexes: BM25 + Qdrant (semantic)

In [22]:
import re
import numpy as np
from typing import Iterable
from qdrant_client import QdrantClient, models as qmodels
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# ============================================================
# STEP 1: Build Rich Document Text
# ============================================================

def build_rich_document_text(row_or_metadata: dict) -> str:
    """
    Build RICH document text for better BM25 and semantic search.
    Include all important searchable fields.
    """
    get = lambda k: str(row_or_metadata.get(k, "")).strip()

    parts = []
    if get("ProductTitle"):
        parts.append(f"ProductTitle: {get('ProductTitle')}")
    if get("Colour"):
        parts.append(f"Colour: {get('Colour')}")
    if get("Gender"):
        parts.append(f"Gender: {get('Gender')}")
    if get("ProductType"):
        parts.append(f"ProductType: {get('ProductType')}")
    if get("SubCategory"):
        parts.append(f"SubCategory: {get('SubCategory')}")
    if get("Usage"):
        parts.append(f"Usage: {get('Usage')}")
    if get("Size"):
        parts.append(f"Size: {get('Size')}")
    if get("ProductId"):
        parts.append(f"ProductId: {get('ProductId')}")

    return " | ".join(parts)

# Update all documents with rich text
for doc in documents:
    doc["text"] = build_rich_document_text(doc["metadata"])

print(f"✅ Built {len(documents)} rich documents")
print(f"📄 Sample: {documents[0]['text'][:100]}...")

# ============================================================
# STEP 2: Preprocessing Functions
# ============================================================

def preprocess_for_bm25(text: str) -> list[str]:
    """
    Improved BM25 tokenization:
    - Lowercase
    - Remove punctuation
    - Remove stop words (commented out for now)
    """
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = text.split()

    # Stop words commented out for now
    # stop_words = {
    #     'a', 'an', 'the', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    #     'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should',
    #     'to', 'of', 'in', 'for', 'on', 'with', 'at', 'by', 'from', 'as', 'into',
    #     'and', 'but', 'or', 'nor', 'so', 'yet', 'both', 'either', 'neither',
    #     'not', 'only', 'own', 'same', 'than', 'too', 'very', 'just',
    #     'i', 'me', 'my', 'we', 'our', 'you', 'your', 'he', 'him', 'his',
    #     'she', 'her', 'it', 'its', 'they', 'them', 'their',
    #     'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those',
    #     'am', 'any', 'each', 'few', 'more', 'most', 'other', 'some',
    #     'hey', 'hi', 'hello', 'please', 'thanks', 'thank',
    #     'looking', 'want', 'need', 'like', 'show', 'find', 'get',
    #     'daughter', 'son', 'kid', 'child',
    # }
    # tokens = [t for t in tokens if t and t not in stop_words and len(t) > 1]

    # Simple filtering: just remove single-character tokens
    tokens = [t for t in tokens if t and len(t) > 1]
    return tokens

# ============================================================
# STEP 3: Filter Functions
# ============================================================

def _build_qdrant_filter(filters: dict | None) -> qmodels.Filter | None:
    if not filters:
        return None
    must = []
    for key, value in filters.items():
        if value is None or value == "":
            continue
        must.append(qmodels.FieldCondition(key=key, match=qmodels.MatchValue(value=value)))
    return qmodels.Filter(must=must) if must else None


def _passes_metadata_filter(metadata: dict, filters: dict | None) -> bool:
    if not filters:
        return True
    for key, value in filters.items():
        if value is None or value == "":
            continue
        if str(metadata.get(key, "")) != str(value):
            return False
    return True

print("✅ Preprocessing and filter functions defined")

✅ Built 2906 rich documents
📄 Sample: ProductTitle: Gini and Jony Girls Knit White Top | Colour: White | Gender: Girls | ProductType: Tops...
✅ Preprocessing and filter functions defined


In [24]:
# ============================================================
# STEP 4: Build Indexes (BM25 + Qdrant)
# ============================================================

EMBED_MODEL_NAME = os.environ.get("EMBED_MODEL_NAME", "sentence-transformers/all-MiniLM-L6-v2")

# Initialize embedder
embedder = SentenceTransformer(EMBED_MODEL_NAME)
vector_size = embedder.get_sentence_embedding_dimension()
print(f"✅ Embedder loaded: {EMBED_MODEL_NAME} (dim={vector_size})")

# --- BM25 Index (with improved preprocessing) ---
print("\n🔧 Building BM25 index...")
corpus_tokens = [preprocess_for_bm25(doc["text"]) for doc in documents]
bm25 = BM25Okapi(corpus_tokens)
print(f"✅ BM25 index: {len(corpus_tokens)} documents")
print(f"   Sample tokens: {corpus_tokens[0][:10]}...")

# --- Qdrant Index (semantic) ---
print("\n🔧 Building Qdrant (semantic) index...")

# Close existing connection if any
try:
    qdrant.close()
except:
    pass

# Use in-memory Qdrant to avoid lock issues (simpler for notebooks)
# Change to path="qdrant_local" if you need persistence
qdrant = QdrantClient(":memory:")
collection_name = "fashion_catalog"

# Create collection (no need to delete for in-memory)
qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=qmodels.VectorParams(size=vector_size, distance=qmodels.Distance.COSINE),
)

# Index documents in batches
batch_size = 256
for i in range(0, len(documents), batch_size):
    batch = documents[i : i + batch_size]
    texts = [d["text"] for d in batch]
    embeddings = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

    points = [
        qmodels.PointStruct(
            id=d["id"],
            vector=embeddings[j].tolist(),
            payload={"text": d["text"], **d["metadata"]},
        )
        for j, d in enumerate(batch)
    ]
    qdrant.upsert(collection_name=collection_name, points=points)

print(f"✅ Qdrant index: {len(documents)} documents (in-memory)")
print(f"\n📊 Indexes ready!")

✅ Embedder loaded: sentence-transformers/all-MiniLM-L6-v2 (dim=384)

🔧 Building BM25 index...
✅ BM25 index: 2906 documents
   Sample tokens: ['producttitle', 'gini', 'and', 'jony', 'girls', 'knit', 'white', 'top', 'colour', 'white']...

🔧 Building Qdrant (semantic) index...
✅ Qdrant index: 2906 documents (in-memory)

📊 Indexes ready!


## Retrieval utilities (hybrid + metadata filtering)

In [6]:
# ============================================================
# STEP 5: Search Functions (using improved preprocessing)
# ============================================================

def bm25_search(query: str, k: int = 10, filters: dict | None = None):
    """
    BM25 lexical search with IMPROVED preprocessing.
    """
    # Use the same preprocessing as indexing
    tokens = preprocess_for_bm25(query)

    if not tokens:
        return []

    scores = bm25.get_scores(tokens)
    scored = []

    for idx, score in enumerate(scores):
        doc = documents[idx]
        if not _passes_metadata_filter(doc["metadata"], filters):
            continue
        scored.append({
            "id": doc["id"],
            "text": doc["text"],
            "metadata": doc["metadata"],
            "score": float(score),
            "source": "bm25",
        })

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:k]


def semantic_search(query: str, k: int = 10, filters: dict | None = None):
    """
    Semantic (vector) search using Qdrant.
    """
    vector = embedder.encode([query], normalize_embeddings=True)[0]
    q_filter = _build_qdrant_filter(filters)

    try:
        resp = qdrant.query_points(
            collection_name=collection_name,
            query=vector.tolist(),
            limit=k,
            query_filter=q_filter,
        )
        hits = getattr(resp, "points", resp)
    except AttributeError:
        hits = qdrant.search(
            collection_name=collection_name,
            query_vector=vector.tolist(),
            limit=k,
            query_filter=q_filter,
        )

    results = []
    for hit in hits:
        payload = hit.payload or {}
        results.append({
            "id": hit.id,
            "text": payload.get("text", ""),
            "metadata": {k: v for k, v in payload.items() if k != "text"},
            "score": float(hit.score),
            "source": "semantic",
        })
    return results


def _rrf_fuse(result_lists: Iterable[list[dict]], k: int = 10, rrf_k: int = 60):
    """Reciprocal Rank Fusion to combine multiple result lists."""
    fused = {}
    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            doc_id = item["id"]
            fused.setdefault(doc_id, {**item, "rrf_score": 0.0})
            fused[doc_id]["rrf_score"] += 1.0 / (rrf_k + rank)
    merged = list(fused.values())
    merged.sort(key=lambda x: x["rrf_score"], reverse=True)
    return merged[:k]


def hybrid_retrieve(query: str, k: int = 10, filters: dict | None = None):
    """Hybrid retrieval: BM25 + Semantic with RRF fusion."""
    bm25_results = bm25_search(query, k=k, filters=filters)
    semantic_results = semantic_search(query, k=k, filters=filters)
    fused = _rrf_fuse([bm25_results, semantic_results], k=k)
    return fused


def rerank_cross_encoder(query: str, docs: list[dict], enabled: bool = False):
    """Optional cross-encoder re-ranking."""
    if not enabled:
        return docs
    try:
        from sentence_transformers import CrossEncoder
    except Exception:
        print("CrossEncoder not available, skipping rerank")
        return docs

    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    pairs = [[query, d["text"]] for d in docs]
    scores = reranker.predict(pairs)
    for i, s in enumerate(scores):
        docs[i]["rerank_score"] = float(s)
    docs.sort(key=lambda x: x.get("rerank_score", 0.0), reverse=True)
    return docs

print("✅ Search functions defined (with improved preprocessing)")

# Quick test
print("\n🔍 Quick test: 'white tops girls Gini Jony'")
test_results = bm25_search("white tops girls Gini Jony", k=3)
for r in test_results:
    print(f"   Score: {r['score']:.2f} | {r['metadata'].get('ProductId')} | {r['metadata'].get('ProductTitle', '')[:40]}...")

✅ Search functions defined (with improved preprocessing)

🔍 Quick test: 'white tops girls Gini Jony'
   Score: 9.66 | 31173 | Gini and Jony Girls Printed White Top...
   Score: 9.66 | 45568 | Gini and Jony Girls Knit White Top...
   Score: 9.66 | 31160 | Gini and Jony Girls Printed White Top...


### Inline notes for retrieval utilities

- `_build_qdrant_filter()` builds exact‑match filters from metadata; this mirrors SQL `WHERE key = value`.
- `_passes_metadata_filter()` applies the same filter to BM25 results (since BM25 is in‑memory).
- `bm25_search()` is lexical search; it returns a list of scored docs.
- `semantic_search()` is vector search against Qdrant and returns scored docs.
- `_rrf_fuse()` combines rankings using Reciprocal Rank Fusion (RRF).
- `hybrid_retrieve()` runs BM25 + semantic and fuses results.
- `rerank_cross_encoder()` is optional non‑LLM reranking with a cross‑encoder.

## RAG fusion helpers

In [7]:
from langchain_core.messages import HumanMessage


def generate_query_variations(query: str, llm=None, n: int = 3) -> list[str]:
    if llm is None:
        # Heuristic fallback: simple variations
        base = query.strip()
        return [base, base.lower(), base.replace("?", "")] [:n]

    prompt = (
        "Generate {n} diverse search queries that are paraphrases of the user query. "
        "Return one query per line, no numbering.\n\nUser query: {query}"
    )
    msg = HumanMessage(content=prompt.format(n=n, query=query))
    resp = llm.invoke([msg])
    lines = [ln.strip() for ln in resp.content.splitlines() if ln.strip()]
    # Ensure original query included
    if query not in lines:
        lines.insert(0, query)
    return lines[:n]


def rag_fusion_retrieve(query: str, k: int = 10, filters: dict | None = None, llm=None):
    variations = generate_query_variations(query, llm=llm, n=3)
    all_results = [hybrid_retrieve(q, k=k, filters=filters) for q in variations]
    fused = _rrf_fuse(all_results, k=k)
    return fused, variations

## Reflection / relevance check

In [8]:
def _snippet(text: str, max_chars: int = 300) -> str:
    text = text or ""
    return text if len(text) <= max_chars else (text[: max_chars - 20] + "... [truncated]")


def check_relevance(query: str, docs: list[dict], llm=None, threshold: float = 0.35) -> dict:
    if not docs:
        return {"is_relevant": False, "reason": "no_docs"}

    if llm is None:
        # Heuristic fallback: rely on top fused score
        top_score = docs[0].get("rrf_score", docs[0].get("score", 0.0))
        return {"is_relevant": top_score >= threshold, "reason": f"heuristic_score={top_score:.3f}"}

    context = "\n\n".join(_snippet(d["text"]) for d in docs[:3])
    prompt = (
        "You are verifying whether the retrieved context is relevant to the user query. "
        "Answer with YES or NO, then a short reason.\n\n"
        f"Query: {query}\n\nContext:\n{context}\n"
    )
    msg = HumanMessage(content=prompt)
    resp = llm.invoke([msg]).content.strip()
    is_relevant = resp.upper().startswith("YES")
    return {"is_relevant": is_relevant, "reason": resp}

## LangGraph flow (retrieve → reflect → fuse → reflect)

In [9]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class RetrievalState(TypedDict, total=False):
    query: str
    filters: dict
    use_fusion: bool
    fusion_queries: list[str]
    retrieved_docs: list[dict]
    is_relevant: bool
    relevance_reason: str
    final_payload: dict


def retrieve_node(state: RetrievalState) -> RetrievalState:
    query = state["query"]
    filters = state.get("filters")

    if state.get("use_fusion"):
        docs, variations = rag_fusion_retrieve(query, k=10, filters=filters, llm=light_llm)
        state["fusion_queries"] = variations
    else:
        docs = hybrid_retrieve(query, k=10, filters=filters)

    docs = rerank_cross_encoder(query, docs, enabled=False)

    state["retrieved_docs"] = docs
    return state


def reflect_node(state: RetrievalState) -> RetrievalState:
    query = state["query"]
    docs = state.get("retrieved_docs", [])

    result = check_relevance(query, docs, llm=good_llm)
    state["is_relevant"] = result["is_relevant"]
    state["relevance_reason"] = result["reason"]
    return state


def enable_fusion_node(state: RetrievalState) -> RetrievalState:
    state["use_fusion"] = True
    return state


def finalize_node(state: RetrievalState) -> RetrievalState:
    # Placeholder for the generator in the full agent
    if state.get("is_relevant"):
        state["final_payload"] = {
            "status": "ready_for_generation",
            "docs": state.get("retrieved_docs", [])[:5],
        }
    else:
        state["final_payload"] = {
            "status": "no_data",
            "message": "No relevant data found after fusion.",
        }
    return state


def should_fuse(state: RetrievalState) -> str:
    if state.get("is_relevant"):
        return "finalize"
    if not state.get("use_fusion"):
        return "fuse"
    return "finalize"


graph = StateGraph(RetrievalState)

graph.add_node("retrieve", retrieve_node)
graph.add_node("reflect", reflect_node)
graph.add_node("enable_fusion", enable_fusion_node)
graph.add_node("finalize", finalize_node)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "reflect")

graph.add_conditional_edges(
    "reflect",
    should_fuse,
    {
        "fuse": "enable_fusion",
        "finalize": "finalize",
    },
)

graph.add_edge("enable_fusion", "retrieve")
graph.add_edge("finalize", END)

retrieval_graph = graph.compile()
print(retrieval_graph)

## Example run

In [10]:
query = "Do you have a pink top for girls from Gini and Jony?"
filters = {
    "Gender": "Girls",
    "Colour": "Pink",
}

result = retrieval_graph.invoke({
    "query": query,
    "filters": filters,
    "use_fusion": False,
})

print("Relevance:", result.get("is_relevant"), result.get("relevance_reason"))
print("Fusion queries:", result.get("fusion_queries"))
print("Final payload status:", result.get("final_payload", {}).get("status"))

# Show top retrieved docs with FULL metadata
print("\n" + "=" * 60)
print("RETRIEVED DOCUMENTS (Full Data)")
print("=" * 60)

for i, d in enumerate((result.get("retrieved_docs") or [])[:5], 1):
    print(f"\n📄 Document {i}")
    print(f"   ID: {d.get('id')}")
    print(f"   Text: {d.get('text')}")
    print(f"   Score: {d.get('rrf_score', d.get('score', 'N/A')):.4f}")
    print(f"   Source: {d.get('source', 'hybrid')}")
    print(f"   Metadata:")
    for key, value in d.get("metadata", {}).items():
        print(f"      - {key}: {value}")

Relevance: True YES. The context provides information about pink tops for girls from the brand Gini and Jony, which directly matches the user's query.
Fusion queries: None
Final payload status: ready_for_generation

RETRIEVED DOCUMENTS (Full Data)

📄 Document 1
   ID: 55
   Text: ProductTitle: Gini and Jony Girls Pink Top | Colour: Pink | Gender: Girls | ProductType: Tops | SubCategory: Topwear | Usage: Casual | Size: Medium | ProductId: 34008
   Score: 0.0325
   Source: bm25
   Metadata:
      - ProductId: 34008
      - Gender: Girls
      - Category: Apparel
      - SubCategory: Topwear
      - ProductType: Tops
      - Colour: Pink
      - Usage: Casual
      - ProductTitle: Gini and Jony Girls Pink Top
      - Image: 34008.jpg
      - ImageURL: https://assets.myntassets.com/v1/images/style/properties/bdb2509ab5b31e6e6fde3f1f1e28f5bf_images.jpg
      - Size: Medium

📄 Document 2
   ID: 351
   Text: ProductTitle: Gini and Jony Girls Pink Top | Colour: Pink | Gender: Girls | ProductTy

## Retrieval Evaluation

This section allows you to evaluate retrieval quality and compare different strategies:
- **Hybrid Retrieval** (BM25 + Semantic)
- **RAG Fusion** (Multi-query + RRF)
- **With/Without Re-ranking**

Metrics computed:
- **Recall@k**: Did we retrieve the relevant document?
- **Precision@k**: How many retrieved docs are relevant?
- **MRR**: Mean Reciprocal Rank (quality of top result)
- **MAP**: Mean Average Precision (ranking quality)
- **NDCG**: Normalized Discounted Cumulative Gain

In [25]:
### Load Evaluation Queries

import ast
from pathlib import Path

def load_evaluation_queries(csv_path: str = "/retrieval_evaluation_queries.csv", use_filters: bool = True):
    """
    Load evaluation queries from CSV file.

    Args:
        csv_path: Path to the evaluation CSV
        use_filters: If True, extract Gender/Colour filters from context (more realistic).
                    If False, run without filters (harder, tests pure retrieval).
    """
    csv_file = Path(csv_path)
    if not csv_file.exists():
        print(f"⚠️ {csv_path} not found. Using default test queries.")
        return [
            {
                "query": "Do you have pink tops for girls?",
                "ground_truth_product_ids": ["23623", "31120"],
                "ground_truth_text": "Pink tops for girls",
                "filters": {"Gender": "Girls", "Colour": "Pink"} if use_filters else {}
            },
        ]

    df_eval = pd.read_csv(csv_file)
    eval_queries = []

    for _, row in df_eval.iterrows():
        gt_text = row["ground_truth"]
        import re
        product_ids = re.findall(r"ProductId[:\s]+(\d+)", gt_text)

        # Parse contexts to extract expected ProductIds AND metadata
        contexts_str = row.get("contexts", "[]")
        try:
            contexts = ast.literal_eval(contexts_str) if isinstance(contexts_str, str) else contexts_str
        except:
            contexts = [contexts_str] if contexts_str else []

        # Extract ProductIds and filters from contexts
        filters = {}
        for ctx in contexts:
            ctx_str = str(ctx)
            ctx_ids = re.findall(r"ProductId[:\s]+(\d+)", ctx_str)
            product_ids.extend(ctx_ids)

            # Extract filters from context metadata
            if use_filters:
                gender_match = re.search(r"Gender[:\s]+(\w+)", ctx_str)
                colour_match = re.search(r"Colour[:\s]+(\w+)", ctx_str)
                if gender_match:
                    filters["Gender"] = gender_match.group(1)
                if colour_match:
                    filters["Colour"] = colour_match.group(1)

        product_ids = list(set(product_ids))

        eval_queries.append({
            "query": row["query"],
            "ground_truth_product_ids": product_ids,
            "ground_truth_text": gt_text,
            "filters": filters,
        })

    filter_status = "WITH filters" if use_filters else "WITHOUT filters"
    print(f"✅ Loaded {len(eval_queries)} evaluation queries {filter_status}")
    return eval_queries

# Load evaluation queries WITH filters (realistic scenario)
eval_queries = load_evaluation_queries(use_filters=True)
print(f"\nSample query: {eval_queries[0]['query']}")
print(f"Expected ProductIds: {eval_queries[0]['ground_truth_product_ids']}")
print(f"Filters: {eval_queries[0]['filters']}")

# Also load WITHOUT filters for comparison
eval_queries_no_filter = load_evaluation_queries(use_filters=False)
print(f"\n(Also loaded {len(eval_queries_no_filter)} queries without filters for comparison)")

✅ Loaded 10 evaluation queries WITH filters

Sample query: Hey, do you have any white tops for girls from Gini and Jony in a large size?
Expected ProductIds: ['42419']
Filters: {'Gender': 'Girls', 'Colour': 'White'}
✅ Loaded 10 evaluation queries WITHOUT filters

(Also loaded 10 queries without filters for comparison)


In [26]:
### Evaluation Metrics Utilities

import numpy as np
from typing import Callable

def calculate_retrieval_metrics(
    retrieved_docs: list[dict],
    ground_truth_ids: list[str],
    k: int = 10
) -> dict:
    """
    Calculate standard retrieval metrics.

    Args:
        retrieved_docs: List of retrieved documents with 'id' and 'metadata' fields
        ground_truth_ids: List of expected ProductIds (as strings)
        k: Number of top results to consider

    Returns:
        Dictionary of metrics
    """
    # Get retrieved ProductIds
    retrieved_ids = []
    for doc in retrieved_docs[:k]:
        # Try to get ProductId from metadata or id
        product_id = str(doc.get("metadata", {}).get("ProductId", doc.get("id", "")))
        retrieved_ids.append(product_id)

    # Convert ground truth to set for fast lookup
    gt_set = set(str(pid) for pid in ground_truth_ids)

    # Hit@k: Did we retrieve at least one relevant document?
    hits = [1 if rid in gt_set else 0 for rid in retrieved_ids]
    hit_at_k = 1.0 if any(hits) else 0.0

    # Recall@k: Proportion of ground truth docs retrieved
    retrieved_relevant = len(set(retrieved_ids) & gt_set)
    recall_at_k = retrieved_relevant / len(gt_set) if gt_set else 0.0

    # Precision@k: Proportion of retrieved docs that are relevant
    precision_at_k = retrieved_relevant / len(retrieved_ids) if retrieved_ids else 0.0

    # MRR: Reciprocal Rank of first relevant document
    mrr = 0.0
    for i, rid in enumerate(retrieved_ids, 1):
        if rid in gt_set:
            mrr = 1.0 / i
            break

    # MAP: Mean Average Precision
    relevant_count = 0
    precision_sum = 0.0
    for i, rid in enumerate(retrieved_ids, 1):
        if rid in gt_set:
            relevant_count += 1
            precision_sum += relevant_count / i
    map_score = precision_sum / len(gt_set) if gt_set else 0.0

    # NDCG@k: Normalized Discounted Cumulative Gain
    dcg = 0.0
    for i, rid in enumerate(retrieved_ids, 1):
        if rid in gt_set:
            dcg += 1.0 / np.log2(i + 1)

    # Ideal DCG (all relevant docs at top)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(gt_set), k) + 1))
    ndcg_at_k = dcg / idcg if idcg > 0 else 0.0

    return {
        "hit@k": hit_at_k,
        "recall@k": recall_at_k,
        "precision@k": precision_at_k,
        "mrr": mrr,
        "map": map_score,
        "ndcg@k": ndcg_at_k,
        "retrieved_ids": retrieved_ids,
        "ground_truth_ids": list(gt_set),
        "num_retrieved": len(retrieved_ids),
        "num_relevant_retrieved": retrieved_relevant,
    }


def evaluate_retrieval_strategy(
    retrieval_fn: Callable,
    eval_queries: list[dict],
    k: int = 5,
    strategy_name: str = "Strategy"
) -> pd.DataFrame:
    """
    Evaluate a retrieval strategy on all evaluation queries.

    Args:
        retrieval_fn: Function that takes (query, k, filters) and returns list of docs
        eval_queries: List of evaluation query dictionaries
        k: Number of results to retrieve
        strategy_name: Name of the strategy for display

    Returns:
        DataFrame with per-query and aggregate metrics
    """
    results = []

    print(f"\n{'='*60}")
    print(f"Evaluating: {strategy_name} (k={k})")
    print(f"{'='*60}")

    for i, eq in enumerate(eval_queries, 1):
        query = eq["query"]
        gt_ids = eq["ground_truth_product_ids"]
        filters = eq.get("filters", {})

        # Run retrieval
        try:
            docs = retrieval_fn(query, k=k, filters=filters if filters else None)
            metrics = calculate_retrieval_metrics(docs, gt_ids, k=k)
            metrics["query"] = query[:50] + "..." if len(query) > 50 else query
            metrics["strategy"] = strategy_name
            results.append(metrics)

            status = "✅" if metrics["hit@k"] > 0 else "❌"
            print(f"{status} Query {i}/{len(eval_queries)}: MRR={metrics['mrr']:.2f}, Recall={metrics['recall@k']:.2f}")
        except Exception as e:
            print(f"❌ Query {i} failed: {e}")
            results.append({
                "query": query[:50],
                "strategy": strategy_name,
                "hit@k": 0, "recall@k": 0, "precision@k": 0,
                "mrr": 0, "map": 0, "ndcg@k": 0, "error": str(e)
            })

    df = pd.DataFrame(results)
    return df


def print_evaluation_summary(df: pd.DataFrame, strategy_name: str = "Strategy"):
    """Print aggregate metrics summary"""
    print(f"\n{'='*60}")
    print(f"📊 {strategy_name} - EVALUATION SUMMARY")
    print(f"{'='*60}")
    print(f"\nTotal Queries: {len(df)}")
    print(f"\n🎯 Retrieval Metrics (averaged):")
    print(f"  • Hit@k:       {df['hit@k'].mean():.3f}  (found at least 1 relevant)")
    print(f"  • Recall@k:    {df['recall@k'].mean():.3f}  (completeness)")
    print(f"  • Precision@k: {df['precision@k'].mean():.3f}  (accuracy)")
    print(f"  • MRR:         {df['mrr'].mean():.3f}  (first relevant rank)")
    print(f"  • MAP:         {df['map'].mean():.3f}  (ranking quality)")
    print(f"  • NDCG@k:      {df['ndcg@k'].mean():.3f}  (graded relevance)")
    print(f"\n✅ Success Rate: {df['hit@k'].mean()*100:.1f}%")
    print(f"{'='*60}")

print("✅ Evaluation utilities loaded")

✅ Evaluation utilities loaded


In [27]:
### Run Evaluation - Compare Retrieval Strategies

# Define retrieval strategies to compare
def bm25_only(query, k=5, filters=None):
    """BM25 lexical search only"""
    return bm25_search(query, k=k, filters=filters)

def semantic_only(query, k=5, filters=None):
    """Semantic search only"""
    return semantic_search(query, k=k, filters=filters)

def hybrid_no_rerank(query, k=5, filters=None):
    """Hybrid (BM25 + Semantic) without re-ranking"""
    return hybrid_retrieve(query, k=k, filters=filters)

def hybrid_with_rerank(query, k=5, filters=None):
    """Hybrid with cross-encoder re-ranking"""
    docs = hybrid_retrieve(query, k=k*2, filters=filters)  # Get more, then rerank
    return rerank_cross_encoder(query, docs, enabled=True)[:k]

def rag_fusion_retrieval(query, k=5, filters=None):
    """RAG Fusion (multi-query + RRF)"""
    docs, _ = rag_fusion_retrieve(query, k=k, filters=filters, llm=light_llm)
    return docs

# Evaluation settings
EVAL_K = 5  # Number of results to retrieve

# Run evaluation for each strategy
all_results = []

# 1. BM25 Only
df_bm25 = evaluate_retrieval_strategy(bm25_only, eval_queries, k=EVAL_K, strategy_name="BM25 Only")
all_results.append(df_bm25)

# 2. Semantic Only
df_semantic = evaluate_retrieval_strategy(semantic_only, eval_queries, k=EVAL_K, strategy_name="Semantic Only")
all_results.append(df_semantic)

# 3. Hybrid (BM25 + Semantic)
df_hybrid = evaluate_retrieval_strategy(hybrid_no_rerank, eval_queries, k=EVAL_K, strategy_name="Hybrid (BM25+Semantic)")
all_results.append(df_hybrid)

# 4. RAG Fusion
df_fusion = evaluate_retrieval_strategy(rag_fusion_retrieval, eval_queries, k=EVAL_K, strategy_name="RAG Fusion")
all_results.append(df_fusion)

print("\n✅ Evaluation complete for all strategies!")


Evaluating: BM25 Only (k=5)
❌ Query 1/10: MRR=0.00, Recall=0.00
✅ Query 2/10: MRR=0.25, Recall=1.00
❌ Query 3/10: MRR=0.00, Recall=0.00
❌ Query 4/10: MRR=0.00, Recall=0.00
✅ Query 5/10: MRR=0.50, Recall=1.00
❌ Query 6/10: MRR=0.00, Recall=0.00
✅ Query 7/10: MRR=0.25, Recall=1.00
✅ Query 8/10: MRR=0.33, Recall=1.00
✅ Query 9/10: MRR=1.00, Recall=1.00
✅ Query 10/10: MRR=0.33, Recall=1.00

Evaluating: Semantic Only (k=5)
❌ Query 1/10: MRR=0.00, Recall=0.00
✅ Query 2/10: MRR=0.25, Recall=1.00
❌ Query 3/10: MRR=0.00, Recall=0.00
✅ Query 4/10: MRR=1.00, Recall=1.00
❌ Query 5/10: MRR=0.00, Recall=0.00
❌ Query 6/10: MRR=0.00, Recall=0.00
❌ Query 7/10: MRR=0.00, Recall=0.00
❌ Query 8/10: MRR=0.00, Recall=0.00
✅ Query 9/10: MRR=1.00, Recall=1.00
✅ Query 10/10: MRR=0.33, Recall=1.00

Evaluating: Hybrid (BM25+Semantic) (k=5)
❌ Query 1/10: MRR=0.00, Recall=0.00
✅ Query 2/10: MRR=0.33, Recall=1.00
❌ Query 3/10: MRR=0.00, Recall=0.00
✅ Query 4/10: MRR=0.33, Recall=1.00
✅ Query 5/10: MRR=0.20, Recall

In [28]:
### Comparison Table - All Strategies

# Combine all results
df_all = pd.concat(all_results, ignore_index=True)

# Create comparison summary
comparison = df_all.groupby("strategy").agg({
    "hit@k": "mean",
    "recall@k": "mean",
    "precision@k": "mean",
    "mrr": "mean",
    "map": "mean",
    "ndcg@k": "mean",
}).round(3)

# Sort by MRR (or your preferred metric)
comparison = comparison.sort_values("mrr", ascending=False)

print("\n" + "=" * 80)
print("📊 RETRIEVAL STRATEGY COMPARISON")
print("=" * 80)
print(f"\nEvaluation: {len(eval_queries)} queries, k={EVAL_K}")
print("\n")

# Display comparison table
display(comparison)

# Find best strategy for each metric
print("\n🏆 Best Strategy per Metric:")
for col in ["hit@k", "recall@k", "precision@k", "mrr", "map", "ndcg@k"]:
    best = comparison[col].idxmax()
    best_val = comparison.loc[best, col]
    print(f"  • {col:12s}: {best} ({best_val:.3f})")

# Calculate improvement from baseline (BM25)
print("\n📈 Improvement over BM25 Baseline:")
if "BM25 Only" in comparison.index:
    baseline = comparison.loc["BM25 Only"]
    for strategy in comparison.index:
        if strategy != "BM25 Only":
            mrr_diff = comparison.loc[strategy, "mrr"] - baseline["mrr"]
            recall_diff = comparison.loc[strategy, "recall@k"] - baseline["recall@k"]
            print(f"  • {strategy}: MRR {mrr_diff:+.3f}, Recall {recall_diff:+.3f}")


📊 RETRIEVAL STRATEGY COMPARISON

Evaluation: 10 queries, k=5




,hit@k,recall@k,precision@k,mrr,map,ndcg@k
strategy,,,,,,
BM25 Only,0.6,0.6,0.20,0.267,0.267,0.349
Semantic Only,0.4,0.4,0.16,0.258,0.258,0.293
Hybrid (BM25+Semantic),0.5,0.5,0.18,0.237,0.237,0.302
RAG Fusion,0.4,0.4,0.16,0.220,0.220,0.265



🏆 Best Strategy per Metric:
  • hit@k       : BM25 Only (0.600)
  • recall@k    : BM25 Only (0.600)
  • precision@k : BM25 Only (0.200)
  • mrr         : BM25 Only (0.267)
  • map         : BM25 Only (0.267)
  • ndcg@k      : BM25 Only (0.349)

📈 Improvement over BM25 Baseline:
  • Semantic Only: MRR -0.009, Recall -0.200
  • Hybrid (BM25+Semantic): MRR -0.030, Recall -0.100
  • RAG Fusion: MRR -0.047, Recall -0.200


In [29]:
### Detailed Per-Query Results

def show_detailed_results(df: pd.DataFrame, strategy_name: str):
    """Show detailed results for a specific strategy"""
    strategy_df = df[df["strategy"] == strategy_name].copy()

    print(f"\n{'='*70}")
    print(f"📋 Detailed Results: {strategy_name}")
    print(f"{'='*70}")

    for i, row in strategy_df.iterrows():
        status = "✅" if row["hit@k"] > 0 else "❌"
        print(f"\n{status} Query: {row['query']}")
        print(f"   Expected: {row.get('ground_truth_ids', 'N/A')}")
        print(f"   Retrieved: {row.get('retrieved_ids', 'N/A')[:5]}")
        print(f"   Metrics: MRR={row['mrr']:.2f}, Recall={row['recall@k']:.2f}, Precision={row['precision@k']:.2f}")

# Show detailed results for best and baseline strategies
best_strategy = comparison.index[0]  # Best by MRR
print_evaluation_summary(df_all[df_all["strategy"] == best_strategy], best_strategy)
show_detailed_results(df_all, best_strategy)

# Also show BM25 baseline for comparison
if "BM25 Only" in comparison.index and best_strategy != "BM25 Only":
    print("\n\n" + "="*70)
    print("📋 Baseline Comparison (BM25 Only)")
    print("="*70)
    show_detailed_results(df_all, "BM25 Only")


📊 BM25 Only - EVALUATION SUMMARY

Total Queries: 10

🎯 Retrieval Metrics (averaged):
  • Hit@k:       0.600  (found at least 1 relevant)
  • Recall@k:    0.600  (completeness)
  • Precision@k: 0.200  (accuracy)
  • MRR:         0.267  (first relevant rank)
  • MAP:         0.267  (ranking quality)
  • NDCG@k:      0.349  (graded relevance)

✅ Success Rate: 60.0%

📋 Detailed Results: BM25 Only

❌ Query: Hey, do you have any white tops for girls from Gin...
   Expected: ['42419']
   Retrieved: ['45568', '31160', '34004', '34163', '34013']
   Metrics: MRR=0.00, Recall=0.00, Precision=0.00

✅ Query: I'm looking for a black top for my daughter. Do yo...
   Expected: ['34009']
   Retrieved: ['18179', '41307', '31121', '34009', '34161']
   Metrics: MRR=0.25, Recall=1.00, Precision=0.20

❌ Query: I'm looking for a blue casual top for my daughter....
   Expected: ['40143']
   Retrieved: ['34152', '39851', '36730', '36727', '40158']
   Metrics: MRR=0.00, Recall=0.00, Precision=0.00

❌ Query: Hi

In [16]:
### Quick Evaluation Function

def quick_evaluate(
    retrieval_fn: Callable,
    strategy_name: str = "Custom Strategy",
    k: int = 5,
    show_details: bool = True
):
    """
    Quick evaluation function to test any retrieval strategy.

    Usage:
        # Define your custom retrieval function
        def my_custom_retrieval(query, k=5, filters=None):
            # Your custom logic here
            return hybrid_retrieve(query, k=k, filters=filters)

        # Evaluate it
        quick_evaluate(my_custom_retrieval, "My Custom Strategy")
    """
    df = evaluate_retrieval_strategy(retrieval_fn, eval_queries, k=k, strategy_name=strategy_name)
    print_evaluation_summary(df, strategy_name)

    if show_details:
        show_detailed_results(df, strategy_name)

    return df

# Example: Quick test of hybrid retrieval
print("="*70)
print("🚀 Quick Evaluate Example - Run this to test your changes:")
print("="*70)
print("""
# Example usage:

# 1. Define your custom retrieval function:
def my_improved_retrieval(query, k=5, filters=None):
    # Add your improvements here
    docs = hybrid_retrieve(query, k=k*2, filters=filters)
    # Maybe add some post-processing
    return docs[:k]

# 2. Run evaluation:
results = quick_evaluate(my_improved_retrieval, "My Improved Version", k=5)

# 3. Compare with baseline:
# The comparison table above shows how different strategies perform
""")

🚀 Quick Evaluate Example - Run this to test your changes:

# Example usage:

# 1. Define your custom retrieval function:
def my_improved_retrieval(query, k=5, filters=None):
    # Add your improvements here
    docs = hybrid_retrieve(query, k=k*2, filters=filters)
    # Maybe add some post-processing
    return docs[:k]

# 2. Run evaluation:
results = quick_evaluate(my_improved_retrieval, "My Improved Version", k=5)

# 3. Compare with baseline:
# The comparison table above shows how different strategies perform



In [17]:
### Optional: Evaluate with Re-ranking (requires CrossEncoder)

# Uncomment and run this cell to test re-ranking
# Note: This will download the cross-encoder model (~100MB) on first run

"""
# Test hybrid with re-ranking
print("Testing Hybrid + Re-ranking...")

def hybrid_reranked(query, k=5, filters=None):
    # Get more candidates, then rerank
    docs = hybrid_retrieve(query, k=k*3, filters=filters)
    return rerank_cross_encoder(query, docs, enabled=True)[:k]

df_rerank = evaluate_retrieval_strategy(
    hybrid_reranked,
    eval_queries,
    k=EVAL_K,
    strategy_name="Hybrid + Re-ranking"
)

print_evaluation_summary(df_rerank, "Hybrid + Re-ranking")

# Add to comparison
comparison_with_rerank = pd.concat([
    comparison,
    df_rerank.groupby("strategy").agg({
        "hit@k": "mean", "recall@k": "mean", "precision@k": "mean",
        "mrr": "mean", "map": "mean", "ndcg@k": "mean"
    }).round(3)
])

print("\\n📊 Updated Comparison (with Re-ranking):")
display(comparison_with_rerank.sort_values("mrr", ascending=False))
"""

print("ℹ️ To enable re-ranking evaluation, uncomment the code above and run this cell.")

ℹ️ To enable re-ranking evaluation, uncomment the code above and run this cell.


In [18]:
### Export Evaluation Results

# Save comparison results to CSV for documentation
output_file = "evaluation_results.csv"

# Add timestamp
from datetime import datetime
comparison_export = comparison.copy()
comparison_export["timestamp"] = datetime.now().isoformat()
comparison_export["num_queries"] = len(eval_queries)
comparison_export["k"] = EVAL_K

# Save
comparison_export.to_csv(output_file)
print(f"✅ Evaluation results saved to: {output_file}")

# Also save detailed results
df_all.to_csv("evaluation_results_detailed.csv", index=False)
print(f"✅ Detailed results saved to: evaluation_results_detailed.csv")

# Print final summary
print("\n" + "="*70)
print("📊 FINAL EVALUATION SUMMARY")
print("="*70)
print(f"\nDate: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Queries: {len(eval_queries)}")
print(f"Top-k: {EVAL_K}")
print(f"\n🏆 Best Overall Strategy: {comparison.index[0]}")
print(f"   MRR: {comparison.iloc[0]['mrr']:.3f}")
print(f"   Recall@k: {comparison.iloc[0]['recall@k']:.3f}")
print(f"\n📈 Strategy Ranking by MRR:")
for i, (strategy, row) in enumerate(comparison.iterrows(), 1):
    print(f"   {i}. {strategy}: {row['mrr']:.3f}")
print("="*70)

✅ Evaluation results saved to: evaluation_results.csv
✅ Detailed results saved to: evaluation_results_detailed.csv

📊 FINAL EVALUATION SUMMARY

Date: 2026-01-30 03:02
Queries: 10
Top-k: 5

🏆 Best Overall Strategy: BM25 Only
   MRR: 0.283
   Recall@k: 0.600

📈 Strategy Ranking by MRR:
   1. BM25 Only: 0.283
   2. RAG Fusion: 0.245
   3. Hybrid (BM25+Semantic): 0.228
   4. Semantic Only: 0.200
